# Consultas Q1–Q4 y pregunta propia 1 (persona A)
Cada consulta: **pregunta, código, resultado e interpretación**. Los pipelines están en `src/consultas_a.py`.

In [7]:
import sys, pprint
from functools import partial
pp = partial(pprint.pprint, sort_dicts=False)   # respeta el orden real de las etapas
sys.path.append("..")
import pandas as pd
from pymongo import MongoClient

BASE = "panteon_prueba"   # cambiar a "panteon" cuando C termine la carga oficial
from src import consultas_a as Q

pd.set_option("display.max_rows", 100)
db = MongoClient("mongodb://localhost:27017")[BASE]
print(db.secuencias.estimated_document_count(), "secuencias |",
      db.especies.estimated_document_count(), "especies")

239946 secuencias | 2790 especies


## Q1. ¿Cuántos TEs hay por clase, por orden y por superfamilia en toda la base?

In [8]:
pp(Q.pipeline_q1())
r1 = Q.q1(db)
display(r1["por_clase"]); display(r1["por_orden"]); display(r1["por_superfamilia"])

[{'$facet': {'por_clase': [{'$group': {'_id': {'te_clase': '$te_clase'},
                                       'n_te': {'$sum': 1}}},
                           {'$sort': {'n_te': -1}}],
             'por_orden': [{'$group': {'_id': {'te_clase': '$te_clase',
                                               'te_orden': '$te_orden'},
                                       'n_te': {'$sum': 1}}},
                           {'$sort': {'n_te': -1}}],
             'por_superfamilia': [{'$group': {'_id': {'te_clase': '$te_clase',
                                                      'te_orden': '$te_orden',
                                                      'te_superfamilia': '$te_superfamilia'},
                                              'n_te': {'$sum': 1}}},
                                  {'$sort': {'n_te': -1}}]}}]


,n_te,te_clase,porcentaje
0,194774,CLASSI,81.17
1,45172,CLASSII,18.83


,n_te,te_clase,te_orden,porcentaje
0,119368,CLASSI,LTR,49.75
1,68382,CLASSI,LINE,28.50
2,36471,CLASSII,TIR,15.20
3,5075,CLASSII,HELITRON,2.12
4,3634,CLASSI,SINE,1.51
5,3456,CLASSII,MITE,1.44
6,2933,CLASSI,DIRS,1.22
7,344,CLASSI,PLE,0.14
8,113,CLASSI,RETROPOSON,0.05
9,112,CLASSII,CRYPTON,0.05


,n_te,te_clase,te_orden,te_superfamilia,porcentaje
0,50849,CLASSI,LTR,COPIA,21.19
1,43963,CLASSI,LTR,GYPSY,18.32
2,23821,CLASSI,LINE,L1,9.93
3,12202,CLASSI,LTR,ERV,5.09
4,11234,CLASSI,LINE,CR1,4.68
5,10962,CLASSI,LINE,L2,4.57
6,10729,CLASSII,TIR,TC1MARINER,4.47
7,10648,CLASSII,TIR,HAT,4.44
8,5983,CLASSII,TIR,TIR,2.49
9,5143,CLASSI,LINE,RTE,2.14


**Esperado con la base completa:** CLASSI 194.774 (81,2 %) y CLASSII 45.172 (18,8 %); 11 órdenes (LTR 119.368, LINE 68.382, TIR 36.471…); 65 superfamilias (COPIA 50.849 y GYPSY 43.963 encabezan).

*Interpretación:* La base está fuertemente sesgada hacia los retrotransposones. La Clase I reúne el 81,2 % de las secuencias, y solo tres órdenes concentran el 93,4 %: LTR (49,7 %), LINE (28,5 %) y TIR (15,2 %). En superfamilias pasa lo mismo: Copia y Gypsy suman casi el 40 %, mientras que 16 superfamilias tienen menos de 100 secuencias. Para el diseño de la base, esto significa que un índice sobre te_orden filtra poco en los órdenes grandes: una consulta por LTR devuelve la mitad de la colección. Además, 11 superfamilias se llaman igual que su orden (LINE/LINE, TIR/TIR…), con 21.544 secuencias en total; en la práctica, eso indica que la superfamilia no se determinó.

## Q2. ¿Cuántos TEs tiene cada especie, desglosados por clase, orden y superfamilia?

In [9]:
pp(Q.pipeline_q2())
r2 = Q.q2(db)
print(len(r2), "filas (especie × clasificación)")
display(r2.head(20))
display(Q.q2(db, especie="Malus domestica"))   # ejemplo para una especie

[{'$group': {'_id': {'especie_id': '$especie_id',
                     'te_clase': '$te_clase',
                     'te_orden': '$te_orden',
                     'te_superfamilia': '$te_superfamilia'},
             'n_te': {'$sum': 1}}},
 {'$lookup': {'from': 'especies',
              'localField': '_id.especie_id',
              'foreignField': '_id',
              'as': 'esp'}},
 {'$unwind': '$esp'},
 {'$project': {'_id': 0,
               'especie': '$esp.especie',
               'te_clase': '$_id.te_clase',
               'te_orden': '$_id.te_orden',
               'te_superfamilia': '$_id.te_superfamilia',
               'n_te': 1}},
 {'$sort': {'especie': 1, 'n_te': -1}}]
18513 filas (especie × clasificación)


,n_te,especie,te_clase,te_orden,te_superfamilia
0,1,Abortiporus biennis,CLASSI,LTR,BELPAO
1,58,Abrus pulchellus subsp cantoniensis,CLASSI,LTR,GYPSY
2,58,Abrus pulchellus subsp cantoniensis,CLASSI,LTR,COPIA
3,18,Abrus pulchellus subsp cantoniensis,CLASSI,LTR,CAULIMOVIRUS
4,14,Abrus pulchellus subsp cantoniensis,CLASSII,TIR,HAT
5,8,Abrus pulchellus subsp cantoniensis,CLASSII,TIR,CMC
6,7,Abrus pulchellus subsp cantoniensis,CLASSI,LINE,L1
7,7,Abrus pulchellus subsp cantoniensis,CLASSII,TIR,MULE
8,3,Abrus pulchellus subsp cantoniensis,CLASSII,HELITRON,HELITRON
9,3,Abrus pulchellus subsp cantoniensis,CLASSI,LTR,LARD


,n_te,especie,te_clase,te_orden,te_superfamilia
0,671,Malus domestica,CLASSI,LTR,COPIA
1,398,Malus domestica,CLASSI,LTR,GYPSY
2,144,Malus domestica,CLASSII,TIR,TIR
3,84,Malus domestica,CLASSI,LINE,L1
4,65,Malus domestica,CLASSII,TIR,PIFHARBINGER
5,50,Malus domestica,CLASSII,MITE,MITE
6,46,Malus domestica,CLASSI,LTR,TRIM
7,35,Malus domestica,CLASSI,LTR,LARD
8,34,Malus domestica,CLASSII,TIR,HAT
9,21,Malus domestica,CLASSII,TIR,CMC


*Interpretación:* La diversidad de superfamilias por especie varía mucho. La mediana es de 5 superfamilias, pero Danio rerio tiene 36, 493 especies tienen una sola y 652 tienen TEs de un único orden. La mayoría de las especies tiene bibliotecas pequeñas y poco diversas, sobre todo en Fungi. Como la especie se guarda por referencia, la consulta agrupa primero y después hace el $lookup sobre 18.513 grupos, no sobre 239.946 documentos.

## Q3. ¿Cómo cambia la cantidad de TEs por clase y orden entre reinos? ¿Y entre phyla?

In [10]:
pp(Q.pipeline_q3("reino"))
r3_reino = Q.q3(db, "reino")
display(r3_reino)
display((r3_reino / r3_reino.sum() * 100).round(1))   # % de cada orden dentro de cada reino
r3_filo = Q.q3(db, "filo")
display(r3_filo)

[{'$group': {'_id': {'especie_id': '$especie_id',
                     'te_clase': '$te_clase',
                     'te_orden': '$te_orden'},
             'n': {'$sum': 1}}},
 {'$lookup': {'from': 'especies',
              'localField': '_id.especie_id',
              'foreignField': '_id',
              'as': 'esp'}},
 {'$unwind': '$esp'},
 {'$group': {'_id': {'te_clase': '$_id.te_clase',
                     'te_orden': '$_id.te_orden',
                     'reino': '$esp.reino'},
             'n_te': {'$sum': '$n'}}},
 {'$project': {'_id': 0,
               'te_clase': '$_id.te_clase',
               'te_orden': '$_id.te_orden',
               'reino': '$_id.reino',
               'n_te': 1}},
 {'$sort': {'reino': 1, 'n_te': -1}}]


reino                Animalia  Fungi  Plantae
te_clase te_orden                            
CLASSI   DIRS            2125    762       46
         LINE           44477   6137    17768
         LTR            25382   7937    86049
         PLE              257      0       87
         RETROPOSON       113      0        0
         SINE            3091     91      452
CLASSII  CRYPTON          112      0        0
         HELITRON        1167   1695     2213
         MAVERICK          43     15        0
         MITE            2718     97      641
         TIR            21659   3063    11749

reino                Animalia  Fungi  Plantae
te_clase te_orden                            
CLASSI   DIRS             2.1    3.8      0.0
         LINE            44.0   31.0     14.9
         LTR             25.1   40.1     72.3
         PLE              0.3    0.0      0.1
         RETROPOSON       0.1    0.0      0.0
         SINE             3.1    0.5      0.4
CLASSII  CRYPTON          0.1    0.0      0.0
         HELITRON         1.2    8.6      1.9
         MAVERICK         0.0    0.1      0.0
         MITE             2.7    0.5      0.5
         TIR             21.4   15.5      9.9

filo                 Acanthocephala  Angiosperma  Annelida  Anthocerotophyta  \
te_clase te_orden                                                              
CLASSI   DIRS                     0           14        79                 0   
         LINE                     0        17265      1340                28   
         LTR                      0        84523       227               115   
         PLE                      1            0         0                 0   
         RETROPOSON               0            0         0                 0   
         SINE                     0          451         0                 0   
CLASSII  CRYPTON                  0            0         0                 0   
         HELITRON                 0         2144         7                 0   
         MAVERICK                 0            0         0                 0   
         MITE                     0          637        54                 0   
         TIR                      0        11407       105                 1   

filo                 Arthropoda  Ascomycota  Basidiomycota  \
te_clase te_orden                                            
CLASSI   DIRS               140          42            490   
         LINE             15318        3459           1591   
         LTR               5070        4040           3083   
         PLE                 45           0              0   
         RETROPOSON           0           0              0   
         SINE               102          85              1   
CLASSII  CRYPTON              0           0              0   
         HELITRON           443         473            896   
         MAVERICK            29           0              0   
         MITE               568          57             19   
         TIR               3430        1280           1015   

filo                 Blastocladiomycota  Bryophyta  Charophyta  ...  \
te_clase te_orden                                               ...   
CLASSI   DIRS                         8          0           0  ...   
         LINE                        15         66          29  ...   
         LTR                          6        410          36  ...   
         PLE                          0          0           2  ...   
         RETROPOSON                   0          0           0  ...   
         SINE                         1          0           0  ...   
CLASSII  CRYPTON                      0          0           0  ...   
         HELITRON                     3         29           1  ...   
         MAVERICK                     0          0           0  ...   
         MITE                         0          1           0  ...   
         TIR                          0         63           3  ...   

filo                 Mucoromycota  Nematoda  Oomycota  Pinophyta  \
te_clase te_orden                                                  
CLASSI   DIRS                 196        16         6          0   
         LINE                 844      1133        50         77   
         LTR                  250       523       359        269   
         PLE                    0       136         0          0   
         RETROPOSON             0         0         0          0   
         SINE                   3         6         0          0   
CLASSII  CRYPTON                0         0         0          0   
         HELITRON             271        27        20         12   
         MAVERICK               4         0         9          0   
         MITE                  17         1         1          0   
         TIR                  552       450       153          7   

filo                 Platyhelminthes  Porifera  Pteridophyta  Rhodophyta  \
te_clase te_orden                                                          
CLASSI   DIRS                      0        10             0           9   
         LINE                    416       102            28          10   
         LTR                      88   

**Esperado (reino):** Plantae está dominado por LTR (86.049); Animalia por LINE (44.477) y LTR (25.382); en Fungi domina LTR (7.937), seguido de LINE (6.137). CRYPTON y RETROPOSON solo aparecen en Animalia.

*Interpretación:* Cada reino tiene un perfil distinto:

Plantae: 72 % LTR.
Animalia: LINE (44 %), seguido de LTR (25 %) y TIR (21 %).
Fungi: intermedio, con LTR (40 %) y LINE (31 %). Es el reino con más HELITRON (8,6 %).

Dentro de Animalia hay contrastes fuertes: Arthropoda y Mollusca están dominados por LINE (61 % y 57 %), y Chordata está más repartido.

## Q4. ¿Cuáles son las 10 especies con más TEs y las 10 con menos?

In [11]:
pp(Q.pipeline_q4())
top, bottom, empates = Q.q4(db)
display(top); display(bottom); print(empates)

[{'$group': {'_id': '$especie_id', 'n_te': {'$sum': 1}}},
 {'$facet': {'top': [{'$sort': {'n_te': -1, '_id': 1}},
                     {'$limit': 10},
                     {'$lookup': {'from': 'especies',
                                  'localField': '_id',
                                  'foreignField': '_id',
                                  'as': 'esp'}},
                     {'$unwind': '$esp'},
                     {'$project': {'_id': 0,
                                   'especie': '$esp.especie',
                                   'reino': '$esp.reino',
                                   'n_te': 1}},
                     {'$sort': {'n_te': -1, 'especie': 1}}],
             'bottom': [{'$sort': {'n_te': 1, '_id': 1}},
                        {'$limit': 10},
                        {'$lookup': {'from': 'especies',
                                     'localField': '_id',
                                     'foreignField': '_id',
                                     'as': 'e

,n_te,especie,reino
0,1627,Malus domestica,Plantae
1,1352,Danio rerio,Animalia
2,1294,Callithrix jacchus,Animalia
3,1173,Homininae,Animalia
4,1159,Nicotiana attenuata,Plantae
5,1148,Colobus angolensis,Animalia
6,1129,Chlorocebus sabaeus,Animalia
7,1115,Geotrypetes seraphini,Animalia
8,1103,Sitophilus oryzae,Animalia
9,1090,Oryza sativa,Plantae


,n_te,especie,reino
0,1,Abortiporus biennis,Fungi
1,1,Absidia glauca,Fungi
2,1,Achaetomium macrosporum,Fungi
3,1,Acidothrix acidophila,Fungi
4,1,Acropora acuminata,Animalia
5,1,Acropora nasuta,Animalia
6,1,Agrocybe aegerita,Fungi
7,1,Alternaria arborescens,Fungi
8,1,Alternaria carthami,Fungi
9,1,Alternaria gaisen,Fungi


{'minimo': 1, 'n_especies': 326}


**Esperado:** top encabezado por *Malus domestica* (1.627), *Danio rerio* (1.352) y *Callithrix jacchus* (1.294). En el 4.º lugar aparece **Homininae (1.173)**, que no es una especie sino una subfamilia → conecta con la pregunta propia. El mínimo es 1 TE y lo comparten **326 especies**, así que el "bottom 10" es arbitrario: se desempató por orden alfabético y hay que decirlo.

*Interpretación:* Las 10 con más: las encabeza Malus domestica (1.627), seguida de Danio rerio (1.352) y Callithrix jacchus (1.294). En cuarto lugar aparece Homininae (1.173), que no es una especie sino una subfamilia de primates.
Las 10 con menos: no están bien definidas. El mínimo es 1 TE y lo comparten 326 especies, así que cualquier lista de diez depende del criterio de desempate. Es más honesto reportar cuántas especies tienen un solo TE.

## Propia 1. ¿Cuántas "especies" del CSV son en realidad taxones superiores (nombre de una sola palabra), cuántos TEs aportan y de qué órdenes?

In [12]:
pp(Q.pipeline_propia1())
resumen, por_orden, taxones = Q.propia1(db)
display(resumen); display(por_orden); display(taxones)

[{'$group': {'_id': {'especie_id': '$especie_id', 'te_orden': '$te_orden'},
             'n': {'$sum': 1}}},
 {'$lookup': {'from': 'especies',
              'localField': '_id.especie_id',
              'foreignField': '_id',
              'as': 'esp'}},
 {'$unwind': '$esp'},
 {'$addFields': {'es_taxon_superior': {'$regexMatch': {'input': '$esp.especie',
                                                       'regex': '^\\S+$'}}}},
 {'$facet': {'resumen': [{'$group': {'_id': '$es_taxon_superior',
                                     'n_te': {'$sum': '$n'},
                                     'especies': {'$addToSet': '$_id.especie_id'}}},
                         {'$project': {'_id': 0,
                                       'es_taxon_superior': '$_id',
                                       'n_te': 1,
                                       'n_entradas': {'$size': '$especies'}}},
                         {'$sort': {'es_taxon_superior': -1}}],
             'por_orden': [{'$match': {'es_

,n_te,es_taxon_superior,n_entradas,porcentaje_te
0,3806,True,45,1.59
1,236140,False,2745,98.41


,n_te,te_orden
0,1920,TIR
1,894,LTR
2,478,LINE
3,401,SINE
4,52,HELITRON
5,24,CRYPTON
6,17,PLE
7,15,DIRS
8,5,RETROPOSON


,reino,n_te,taxon
0,Animalia,1173,Homininae
1,Animalia,682,Myomorpha
2,Animalia,125,Monotremata
3,Animalia,90,Durocryptodira
4,Animalia,71,Heliconiini
5,Animalia,59,Simiiformes
6,Animalia,59,Hominidae
7,Animalia,59,Hominoidea
8,Animalia,59,Platyrrhini
9,Animalia,58,Amniota


**Esperado:** 45 entradas (todas de Animalia) con 3.806 TEs (1,59 %). Las mayores son Homininae (1.173) y Myomorpha (682). Por orden dominan TIR (1.920) y LTR (894).

*Interpretación:* estas bibliotecas de Dfam están asignadas a un clado y no a una especie, por lo que el campo `especie` no siempre es una especie; esto afecta las consultas por especie (Q2, Q4) y la herencia de taxonomía (filo, clase…).